# PASO 2 — Limpieza e Integración de Datos (ETL)
## Proyecto: Análisis de Churn en TELCO

**Equipo:** Data & BI

**Objetivo de esta fase:** partir de los 5 CSV de `data/raw/` y, aplicando las
decisiones tomadas durante el EDA (PASO 1), construir **un único dataset limpio,
a nivel de cliente**, listo para el análisis multivariante, el modelado predictivo
y el clustering.

Seguimos los 7 pasos del proceso ETL del documento de requisitos:

1. Homogeneizar nombres de columnas.
2. Conversión de tipos de datos.
3. Tratamiento de duplicados.
4. Tratamiento de valores nulos.
5. Detección y corrección de inconsistencias.
6. Integración de tablas (merge).
7. Generación de variables derivadas.

**Entregable de esta fase:** `data/cleaned/telco_cleaned.csv`.

## 0. Configuración inicial

Cargamos librerías y los 5 datasets en bruto desde `data/raw/`.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 250)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

RAW_PATH = '../data/raw/'
CLEANED_PATH = '../data/cleaned/'

demographics = pd.read_csv(RAW_PATH + 'Telco_customer_churn_demographics.csv')
location     = pd.read_csv(RAW_PATH + 'Telco_customer_churn_location.csv')
population   = pd.read_csv(RAW_PATH + 'Telco_customer_churn_population.csv')
services     = pd.read_csv(RAW_PATH + 'Telco_customer_churn_services.csv')
status       = pd.read_csv(RAW_PATH + 'Telco_customer_churn_status.csv')

for nombre, df in [('demographics', demographics), ('location', location),
                    ('population', population), ('services', services),
                    ('status', status)]:
    print(f"{nombre:<15} -> {df.shape[0]} filas x {df.shape[1]} columnas")

demographics    -> 7043 filas x 9 columnas
location        -> 7043 filas x 8 columnas
population      -> 1671 filas x 3 columnas
services        -> 7043 filas x 30 columnas
status          -> 7043 filas x 8 columnas


## Paso 1 — Homogeneizar nombres de columnas

**Convenio elegido:** `snake_case` — minúsculas, sin espacios ni guiones, separando
palabras con `_`. Es el convenio más habitual en Python/SQL y evita tener que usar
comillas o corchetes para acceder a las columnas (`df.customer_id` en vez de
`df['Customer ID']`).

Aplicamos la transformación a las 5 tablas con la misma cadena de operaciones
(`strip → lower → replace espacios → replace guiones`).

In [2]:
for df in [demographics, location, population, services, status]:
    df.columns = (
        df.columns
        .str.strip() # elimina espacios en blanco al inicio y al final de los nombres de las columnas
        .str.lower() # convierte los nombres de las columnas a minúsculas
        .str.replace(' ', '_') # reemplaza espacios por guiones bajos
        .str.replace('-', '_') # reemplaza guiones por guiones bajos
    )

# list para ver los nombres de las columnas de cada dataframe
print("demographics:", list(demographics.columns))
print("\nlocation:", list(location.columns))
print("\npopulation:", list(population.columns))
print("\nstatus:", list(status.columns))
print("\nservices (primeras 10):", list(services.columns)[:10], "...")

demographics: ['customer_id', 'count', 'gender', 'age', 'under_30', 'senior_citizen', 'married', 'dependents', 'number_of_dependents']

location: ['customer_id', 'count', 'country', 'state', 'city', 'zip_code', 'latitude', 'longitude']

population: ['id', 'zip_code', 'population']

status: ['customer_id', 'count', 'quarter', 'customer_status', 'churn_label', 'churn_value', 'churn_category', 'churn_reason']

services (primeras 10): ['customer_id', 'count', 'quarter', 'referred_a_friend', 'number_of_referrals', 'tenure_in_months', 'offer', 'phone_service', 'avg_monthly_long_distance_charges', 'multiple_lines'] ...


**Limpieza adicional — columnas sin información**

Durante el EDA detectamos varias columnas **constantes o puramente identificadoras
de fila**, que no aportan ninguna información al análisis (varianza = 0) y que,
además, podrían causar problemas en el cálculo del VIF en el PASO 3 (varianza cero
→ división por cero). Las eliminamos ahora, antes de seguir:

| Columna | Tabla(s) | Motivo |
|---|---|---|
| `count` | demographics, location, services, status | Siempre = 1, sin información. |
| `quarter` | services, status | Siempre = "Q3", sin información. |
| `country`, `state` | location | Siempre "United States" / "California" (dataset de un solo estado). |
| `id` | population | Es solo la posición de la fila + 1, redundante con el índice. |

No eliminamos ninguna columna relacionada con el churn (`churn_label`,
`churn_category`, `churn_reason`, `customer_status`) — esas se necesitan para el
PASO 3 (correlación con el churn) y el PASO 6 (dashboard). Su tratamiento de cara al
**modelo predictivo** (data leakage) se abordará explícitamente en el PASO 4, no
aquí (ver pregunta de reflexión 2).

In [3]:
# drop para eliminar columnas que no se van a usar
demographics = demographics.drop(columns=['count'])
location     = location.drop(columns=['count', 'country', 'state'])
population   = population.drop(columns=['id'])
services     = services.drop(columns=['count', 'quarter'])
status       = status.drop(columns=['count', 'quarter'])

for nombre, df in [('demographics', demographics), ('location', location),
                    ('population', population), ('services', services),
                    ('status', status)]:
    print(f"{nombre:<15} -> {df.shape[1]} columnas")

demographics    -> 8 columnas
location        -> 5 columnas
population      -> 2 columnas
services        -> 28 columnas
status          -> 6 columnas


## Paso 2 — Conversión de tipos de datos

En el EDA comprobamos que **todas las columnas numéricas ya vienen con el tipo
correcto** (`int64` / `float64`) en los 5 CSV — no hay importes ni edades
almacenados como texto, por lo que **no es necesario convertir nada** en este
sentido.

Solo dejamos constancia de una decisión conceptual: `zip_code` es de tipo `int64`,
pero **no es una cantidad** sobre la que tenga sentido calcular medias, sumas o
correlaciones — es un **identificador geográfico**. Lo mantenemos como `int64` por
ahora porque es la clave que usaremos para el `merge` con `population` (los tipos
de la clave deben coincidir en ambas tablas), pero **lo excluiremos explícitamente
de cualquier análisis numérico** (correlación, VIF, escalado) en los pasos
siguientes.

In [4]:
print("Tipos de datos por tabla:\n")
for nombre, df in [('demographics', demographics), ('location', location),
                    ('population', population), ('services', services),
                    ('status', status)]:
    print(f"--- {nombre} ---")
    print(df.dtypes.value_counts())
    print()

print("Tipo de 'zip_code' en location:", location['zip_code'].dtype)
print("Tipo de 'zip_code' en population:", population['zip_code'].dtype)

Tipos de datos por tabla:

--- demographics ---
str      6
int64    2
Name: count, dtype: int64

--- location ---
str        2
float64    2
int64      1
Name: count, dtype: int64

--- population ---
int64    2
Name: count, dtype: int64

--- services ---
str        18
float64     6
int64       4
Name: count, dtype: int64

--- status ---
str      5
int64    1
Name: count, dtype: int64

Tipo de 'zip_code' en location: int64
Tipo de 'zip_code' en population: int64


## Paso 3 — Tratamiento de duplicados

En el EDA no encontramos **ninguna fila duplicada** (ni duplicados exactos ni
`customer_id` repetidos) en ninguna de las 5 tablas. Aun así, aplicamos
`drop_duplicates()` de forma **defensiva**: es una operación segura (no elimina
nada si no hay duplicados) y garantiza que el pipeline siga siendo correcto si en
el futuro se recibe una nueva carga de datos con duplicados reales.

Si encontráramos duplicados, la decisión dependería del tipo:
- **Duplicado exacto** (todas las columnas iguales, incluido `customer_id`) → casi
  seguro un error de carga → eliminar.
- **Mismo `customer_id` pero datos distintos** → no sería un duplicado real, sino
  un conflicto de datos que requeriría revisión con negocio (¿qué versión es la
  correcta?).

In [5]:
# Contamos cuántos registros hay en cada dataframe antes de eliminar duplicados
antes = {nombre: len(df) for nombre, df in
         [('demographics', demographics), ('location', location),
          ('population', population), ('services', services), ('status', status)]}

# Eliminamos duplicados con drop_duplicates() y contamos cuántos quedan en cada dataframe
demographics = demographics.drop_duplicates()
location     = location.drop_duplicates()
population    = population.drop_duplicates()
services     = services.drop_duplicates()
status       = status.drop_duplicates()

# Contamos cuántos registros hay en cada dataframe después de eliminar duplicados
despues = {nombre: len(df) for nombre, df in
           [('demographics', demographics), ('location', location),
            ('population', population), ('services', services), ('status', status)]}

for nombre in antes:
    print(f"{nombre:<15} -> antes: {antes[nombre]:>5}  |  despues: {despues[nombre]:>5}  |  eliminadas: {antes[nombre]-despues[nombre]}")

demographics    -> antes:  7043  |  despues:  7043  |  eliminadas: 0
location        -> antes:  7043  |  despues:  7043  |  eliminadas: 0
population      -> antes:  1671  |  despues:  1671  |  eliminadas: 0
services        -> antes:  7043  |  despues:  7043  |  eliminadas: 0
status          -> antes:  7043  |  despues:  7043  |  eliminadas: 0


## Paso 4 — Tratamiento de valores nulos

Según el EDA, solo 4 columnas tienen nulos, y en **los 4 casos el nulo es
información de negocio**, no un dato perdido:

| Columna | Tabla | % nulos | Significado del nulo | Imputación |
|---|---|---|---|---|
| `offer` | services | 55,05% | Cliente sin oferta promocional activa | `'No Offer'` |
| `internet_type` | services | 21,67% | Cliente sin servicio de internet (`internet_service = 'No'`) | `'No Internet'` |
| `churn_category` | status | 73,46% | Cliente que no ha hecho churn | `'No Churn'` |
| `churn_reason` | status | 73,46% | Cliente que no ha hecho churn | `'No Churn'` |

En ningún caso usamos media/mediana/moda, porque **no estamos estimando un valor
desconocido**: estamos codificando explícitamente "ausencia de X" como su propia
categoría. Esto es justo la distinción que pide el consejo del PASO 1: estos nulos
**no son un problema de recogida**, son la forma en que el dataset representa "esto
no aplica a este cliente".

In [6]:
print("Nulos ANTES de la imputacion:")
print("  services.offer:           ", services['offer'].isnull().sum())
print("  services.internet_type:   ", services['internet_type'].isnull().sum())
print("  status.churn_category:    ", status['churn_category'].isnull().sum())
print("  status.churn_reason:      ", status['churn_reason'].isnull().sum())

# fillna para imputar los valores nulos con 'No Offer', 'No Internet' y 'No Churn' respectivamente
services['offer'] = services['offer'].fillna('No Offer')
services['internet_type'] = services['internet_type'].fillna('No Internet')
status['churn_category'] = status['churn_category'].fillna('No Churn')
status['churn_reason'] = status['churn_reason'].fillna('No Churn')

print("\nNulos DESPUES de la imputacion:")
print("  services.offer:           ", services['offer'].isnull().sum())
print("  services.internet_type:   ", services['internet_type'].isnull().sum())
print("  status.churn_category:    ", status['churn_category'].isnull().sum())
print("  status.churn_reason:      ", status['churn_reason'].isnull().sum())

print("\nNulos totales restantes en las 5 tablas:")
for nombre, df in [('demographics', demographics), ('location', location),
                    ('population', population), ('services', services),
                    ('status', status)]:
    print(f"  {nombre:<15} -> {df.isnull().sum().sum()}")

Nulos ANTES de la imputacion:
  services.offer:            3877
  services.internet_type:    1526
  status.churn_category:     5174
  status.churn_reason:       5174

Nulos DESPUES de la imputacion:
  services.offer:            0
  services.internet_type:    0
  status.churn_category:     0
  status.churn_reason:       0

Nulos totales restantes en las 5 tablas:
  demographics    -> 0
  location        -> 0
  population      -> 0
  services        -> 0
  status          -> 0


## Paso 5 — Detección y corrección de inconsistencias

El EDA detectó **tres inconsistencias concretas**, las tres concentradas
principalmente en un bloque final de filas con errores que parecen inyectados
deliberadamente. Las corregimos una a una, documentando la decisión de cada una.

### 5.1 — `gender`: normalizar `M`/`F` → `Male`/`Female`

1.275 registros usan `'M'`/`'F'` en vez de `'Male'`/`'Female'`. Es un simple mapeo
de texto, sin pérdida de información — unificamos a `'Male'`/`'Female'` porque es la
codificación mayoritaria (5.768 de 7.043 filas).

In [7]:
print("gender ANTES:")
print(demographics['gender'].value_counts()) # values_counts() para ver la cantidad de cada valor en la columna

demographics['gender'] = demographics['gender'].replace({'M': 'Male', 'F': 'Female'})

print("\ngender DESPUES:")
print(demographics['gender'].value_counts())

gender ANTES:
gender
Male      2918
Female    2850
F          638
M          637
Name: count, dtype: int64

gender DESPUES:
gender
Male      3555
Female    3488
Name: count, dtype: int64


### 5.2 — `under_30` / `senior_citizen`: recalcular a partir de `age`

Encontramos 109 clientes con `age > 100` (rango 101-119), de los cuales:
- 89 tenían `senior_citizen = 'No'` pese a tener `age >= 65`.
- 20 tenían `under_30 = 'Yes'` pese a tener `age >= 30`.

Para el resto del dataset, estos dos flags **sí eran 100% coherentes** con `age`
(son, de hecho, variables derivadas de la edad). La forma más limpia y consistente
de resolver esto es **recalcular ambos flags directamente a partir de `age` para
las 7.043 filas**, usando las mismas reglas que ya cumplía el 98,5% del dataset:

- `under_30 = 'Yes'` si `age < 30`, si no `'No'`.
- `senior_citizen = 'Yes'` si `age >= 65`, si no `'No'`.

Esto **no corrige la edad en sí** (seguimos teniendo 109 clientes con `age` entre
101 y 119, algo estadísticamente muy improbable), pero sí garantiza que **los flags
sean siempre coherentes con la edad**, que es lo que un analista o un modelo
esperaría. Dejamos constancia de que `age > 100` es una **limitación conocida** del
dataset: en el PASO 3 revisaremos si estos valores distorsionan la correlación o el
VIF, y en el PASO 5 (clustering) si generan outliers visibles en el PCA. Si en algún
momento resultase problemático, se podría revisar entonces (p. ej. capando `age` a
un máximo razonable), pero de momento **no eliminamos esas 109 filas**: harían
perder información válida del resto de columnas (servicios, contrato, revenue...)
de esos clientes.

In [8]:
inc_under30_antes = ((demographics['under_30'] == 'Yes') & (demographics['age'] >= 30)).sum()
inc_senior_antes  = ((demographics['senior_citizen'] == 'No') & (demographics['age'] >= 65)).sum()
print("Inconsistencias ANTES:")
print("  under_30='Yes' con age>=30:     ", inc_under30_antes)
print("  senior_citizen='No' con age>=65:", inc_senior_antes)

# np.where para crear las columnas under_30 y senior_citizen a partir de age, con condiciones lógicas para asignar 'Yes' o 'No'
# Se reemplazan las columnas under_30 y senior_citizen para corregir las inconsistencias detectadas, asignando 'Yes' o 'No' según la edad
demographics['under_30'] = np.where(demographics['age'] < 30, 'Yes', 'No')
demographics['senior_citizen'] = np.where(demographics['age'] >= 65, 'Yes', 'No')

inc_under30_despues = ((demographics['under_30'] == 'Yes') & (demographics['age'] >= 30)).sum()
inc_senior_despues  = ((demographics['senior_citizen'] == 'No') & (demographics['age'] >= 65)).sum()
print("\nInconsistencias DESPUES:")
print("  under_30='Yes' con age>=30:     ", inc_under30_despues)
print("  senior_citizen='No' con age>=65:", inc_senior_despues)

print("\nClientes con age > 100 (limitacion conocida, se mantienen):", (demographics['age'] > 100).sum())

Inconsistencias ANTES:
  under_30='Yes' con age>=30:      20
  senior_citizen='No' con age>=65: 89

Inconsistencias DESPUES:
  under_30='Yes' con age>=30:      0
  senior_citizen='No' con age>=65: 0

Clientes con age > 100 (limitacion conocida, se mantienen): 109


### 5.3 — `monthly_charge`: corregir signo negativo

120 clientes tienen `monthly_charge` negativo (entre -1 y -10), un valor imposible
para un cargo mensual. Al comparar con `total_charges` y `tenure_in_months` no
encontramos una escala alternativa coherente — todo apunta a un **simple error de
signo**. Aplicamos `abs()`, que corrige el problema sin alterar la magnitud del cargo,
que es la parte de la información que sí parece correcta.

In [9]:
n_negativos = (services['monthly_charge'] < 0).sum()
print("monthly_charge negativos ANTES:", n_negativos)
print("Minimo ANTES:", services['monthly_charge'].min())

# Reemplazamos los valores negativos de monthly_charge por su valor absoluto usando abs(), 
# para corregir los errores detectados en la columna monthly_charge
services['monthly_charge'] = services['monthly_charge'].abs()

print("\nmonthly_charge negativos DESPUES:", (services['monthly_charge'] < 0).sum())
print("Minimo DESPUES:", services['monthly_charge'].min())

monthly_charge negativos ANTES: 120
Minimo ANTES: -10.0

monthly_charge negativos DESPUES: 0
Minimo DESPUES: 1.0


## Paso 6 — Integración de tablas (merge)

**Estructura relacional** (confirmada en el EDA):

- `demographics`, `location`, `services` y `status` están **a nivel de cliente** y
  comparten exactamente el mismo conjunto de 7.043 `customer_id`.
- `population` está **a nivel de código postal** (`zip_code`), con 1.671 filas. Los
  1.626 `zip_code` distintos que aparecen en `location` están **todos** presentes en
  `population`.

**Estrategia de merge:**

1. Encadenamos `demographics → services → location → status` usando
   `customer_id` como clave, con `how='left'` tomando `demographics` como tabla
   base.
2. Unimos el resultado con `population` usando `zip_code`, también con
   `how='left'`.

**¿Por qué `left` y no `inner`?** Dado que las 4 tablas de cliente comparten
exactamente el mismo conjunto de `customer_id` (y los `zip_code` de `location`
están todos en `population`), en este dataset concreto **`inner` y `left` producen
el mismo resultado** (lo comprobamos más abajo). Elegimos `left` porque es la opción
**defensiva**: si en una futura carga apareciera un cliente sin alguna tabla
asociada (p. ej. un cliente nuevo sin fila aún en `status`), `inner` lo **eliminaría
silenciosamente** del dataset (perderíamos un cliente sin darnos cuenta), mientras
que `left` lo conserva con `NaN` en las columnas que falten, dejando el problema
**visible** para tratarlo explícitamente.

In [10]:
telco_df = (
    demographics
    .merge(services, on='customer_id', how='left')
    .merge(location, on='customer_id', how='left')
    .merge(status, on='customer_id', how='left')
)

print("Shape tras el merge de las 4 tablas a nivel de cliente:", telco_df.shape)
print("Nulos generados por este merge:", telco_df.isnull().sum().sum())

Shape tras el merge de las 4 tablas a nivel de cliente: (7043, 44)
Nulos generados por este merge: 0


In [11]:
# population se une por zip_code, no por customer_id
telco_df = telco_df.merge(population, on='zip_code', how='left')

print("Shape tras el merge con population (por zip_code):", telco_df.shape)
print("Nulos en 'population' tras el merge:", telco_df['population'].isnull().sum())

Shape tras el merge con population (por zip_code): (7043, 45)
Nulos en 'population' tras el merge: 0


In [12]:
# Verificacion: inner vs left dan el mismo resultado en este dataset
inner_check = (
    demographics
    .merge(services, on='customer_id', how='inner')
    .merge(location, on='customer_id', how='inner')
    .merge(status, on='customer_id', how='inner')
    .merge(population, on='zip_code', how='inner')
)
print("Filas con left join:", len(telco_df))
print("Filas con inner join:", len(inner_check))
print("¿Mismo resultado?", len(telco_df) == len(inner_check))

Filas con left join: 7043
Filas con inner join: 7043
¿Mismo resultado? True


## Paso 7 — Generación de variables derivadas

Creamos **3 variables nuevas** a partir de columnas existentes, pensadas para
aportar valor al análisis y al futuro dashboard de Power BI (que pide una "tabla de
fechas" y distribuciones por antigüedad/edad):

1. **`fecha_alta`** (fecha de alta del cliente): no disponemos de una fecha real de
   alta, pero sí de `tenure_in_months` (antigüedad en meses). Asumimos una
   **fecha de referencia** = fin del Q3 2024 (`2024-09-30`), coherente con
   `quarter = 'Q3'` (columna que eliminamos en el paso 1, pero cuyo valor seguimos
   teniendo en cuenta aquí) y con el rango de `tenure_in_months` (1-72 meses ≈
   hasta 6 años de antigüedad). `fecha_alta = fecha_referencia - tenure_in_months`
   meses, redondeada al primer día del mes. **Esta fecha de referencia es una
   suposición** que debería confirmarse con negocio si se necesita precisión
   exacta; lo importante es que las fechas resultantes son **relativas y
   consistentes entre sí**, lo cual es suficiente para análisis de tendencias
   (altas por año/mes) en el dashboard.

2. **`antiguedad_anios`**: `tenure_in_months / 12`, redondeado a 2 decimales. Es la
   misma información que `tenure_in_months` pero en una unidad más intuitiva para
   negocio ("este cliente lleva 2,5 años con nosotros" se entiende mejor que "30
   meses").

3. **`grupo_edad`**: bucket categórico de `age` (`<30`, `30-44`, `45-59`, `60+`),
   pensado para el Vista 1 del dashboard ("Distribución de clientes por... edad").
   Usa los mismos puntos de corte que `under_30` (30) y `senior_citizen` (65, lo
   absorbemos dentro de `60+`), por lo que es coherente con esas dos columnas.

In [13]:
REFERENCE_DATE = pd.Timestamp('2024-09-30')

# 1. fecha_alta: fecha de referencia menos la antiguedad en meses
# to_timedelta para convertir la antiguedad en meses a días (usando 30.4375 días por mes) 
# y restarla a la fecha de referencia, luego se convierte a periodo mensual y se vuelve a timestamp para tener solo año-mes
telco_df['fecha_alta'] = REFERENCE_DATE - pd.to_timedelta(telco_df['tenure_in_months'] * 30.4375, unit='D')

# to_period y to_timestamp para convertir a periodo mensual y luego a timestamp, quedando solo año-mes en fecha_alta
telco_df['fecha_alta'] = telco_df['fecha_alta'].dt.to_period('M').dt.to_timestamp()

# 2. antiguedad en anios
telco_df['antiguedad_anios'] = (telco_df['tenure_in_months'] / 12).round(2)

# 3. grupo de edad
# bins es la lista de límites para los grupos de edad, con un límite superior alto para incluir edades extremas
bins = [0, 30, 45, 60, 200]
labels = ['<30', '30-44', '45-59', '60+']

# pd.cut para crear la columna grupo_edad a partir de age, usando los bins y labels definidos, 
# con right=False para que el límite derecho no se incluya en el intervalo
telco_df['grupo_edad'] = pd.cut(telco_df['age'], bins=bins, labels=labels, right=False)

print("fecha_alta -> rango:", telco_df['fecha_alta'].min().date(), "a", telco_df['fecha_alta'].max().date())
print("\nantiguedad_anios -> describe:")
print(telco_df['antiguedad_anios'].describe())
print("\ngrupo_edad -> distribucion:")
print(telco_df['grupo_edad'].value_counts())

fecha_alta -> rango: 2018-09-01 a 2024-08-01

antiguedad_anios -> describe:
count   7,043.00
mean        2.70
std         2.05
min         0.08
25%         0.75
50%         2.42
75%         4.58
max         6.00
Name: antiguedad_anios, dtype: float64

grupo_edad -> distribucion:
grupo_edad
30-44    1919
45-59    1885
60+      1858
<30      1381
Name: count, dtype: int64


## Verificación final del dataset integrado

In [14]:
print("Shape final:", telco_df.shape)
print("\nColumnas finales:")
for c in telco_df.columns:
    print(" -", c)

Shape final: (7043, 48)

Columnas finales:
 - customer_id
 - gender
 - age
 - under_30
 - senior_citizen
 - married
 - dependents
 - number_of_dependents
 - referred_a_friend
 - number_of_referrals
 - tenure_in_months
 - offer
 - phone_service
 - avg_monthly_long_distance_charges
 - multiple_lines
 - internet_service
 - internet_type
 - avg_monthly_gb_download
 - online_security
 - online_backup
 - device_protection_plan
 - premium_tech_support
 - streaming_tv
 - streaming_movies
 - streaming_music
 - unlimited_data
 - contract
 - paperless_billing
 - payment_method
 - monthly_charge
 - total_charges
 - total_refunds
 - total_extra_data_charges
 - total_long_distance_charges
 - total_revenue
 - city
 - zip_code
 - latitude
 - longitude
 - customer_status
 - churn_label
 - churn_value
 - churn_category
 - churn_reason
 - population
 - fecha_alta
 - antiguedad_anios
 - grupo_edad


In [15]:
telco_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 48 columns):
 #   Column                             Non-Null Count  Dtype         
---  ------                             --------------  -----         
 0   customer_id                        7043 non-null   str           
 1   gender                             7043 non-null   str           
 2   age                                7043 non-null   int64         
 3   under_30                           7043 non-null   str           
 4   senior_citizen                     7043 non-null   str           
 5   married                            7043 non-null   str           
 6   dependents                         7043 non-null   str           
 7   number_of_dependents               7043 non-null   int64         
 8   referred_a_friend                  7043 non-null   str           
 9   number_of_referrals                7043 non-null   int64         
 10  tenure_in_months                   7043 non-nul

In [16]:
nulos_finales = telco_df.isnull().sum()
print("Columnas con nulos en el dataset final:")
print(nulos_finales[nulos_finales > 0] if nulos_finales.sum() > 0 else "Ninguna - 0 nulos en todo el dataset")

Columnas con nulos en el dataset final:
Ninguna - 0 nulos en todo el dataset


In [17]:
telco_df.head()

,customer_id,gender,age,under_30,senior_citizen,married,dependents,number_of_dependents,referred_a_friend,number_of_referrals,tenure_in_months,offer,phone_service,avg_monthly_long_distance_charges,multiple_lines,internet_service,internet_type,avg_monthly_gb_download,online_security,online_backup,device_protection_plan,premium_tech_support,streaming_tv,streaming_movies,streaming_music,unlimited_data,contract,paperless_billing,payment_method,monthly_charge,total_charges,total_refunds,total_extra_data_charges,total_long_distance_charges,total_revenue,city,zip_code,latitude,longitude,customer_status,churn_label,churn_value,churn_category,churn_reason,population,fecha_alta,antiguedad_anios,grupo_edad
0,8779-QRDMV,Male,78,No,Yes,No,No,0,No,0,1,No Offer,No,0.00,No,Yes,DSL,8,No,No,Yes,No,No,Yes,No,No,Month-to-Month,Yes,Bank Withdrawal,39.65,39.65,0.00,20,0.00,59.65,Los Angeles,90022,34.02,-118.16,Churned,Yes,1,Competitor,Competitor offered more data,68701,2024-08-01,0.08,60+
1,7495-OOKFY,Female,74,No,Yes,Yes,Yes,1,Yes,1,8,Offer E,Yes,48.85,Yes,Yes,Fiber Optic,17,No,Yes,No,No,No,No,No,Yes,Month-to-Month,Yes,Credit Card,80.65,633.30,0.00,0,390.80,"1,024.10",Los Angeles,90063,34.04,-118.19,Churned,Yes,1,Competitor,Competitor made better offer,55668,2024-01-01,0.67,60+
2,1658-BYGOY,Male,71,No,Yes,No,Yes,3,No,0,18,Offer D,Yes,11.33,Yes,Yes,Fiber Optic,52,No,No,No,No,Yes,Yes,Yes,Yes,Month-to-Month,Yes,Bank Withdrawal,95.45,"1,752.55",45.61,0,203.94,"1,910.88",Los Angeles,90065,34.11,-118.23,Churned,Yes,1,Competitor,Competitor made better offer,47534,2023-04-01,1.50,60+
3,4598-XLKNJ,Female,78,No,Yes,Yes,Yes,1,Yes,1,25,Offer C,Yes,19.76,No,Yes,Fiber Optic,12,No,Yes,Yes,No,Yes,Yes,No,Yes,Month-to-Month,Yes,Bank Withdrawal,98.50,"2,514.50",13.43,0,494.00,"2,995.07",Inglewood,90303,33.94,-118.33,Churned,Yes,1,Dissatisfaction,Limited range of services,27778,2022-08-01,2.08,60+
4,4846-WHAFZ,Female,80,No,Yes,Yes,Yes,1,Yes,1,37,Offer C,Yes,6.33,Yes,Yes,Fiber Optic,14,No,No,No,No,No,No,No,Yes,Month-to-Month,Yes,Bank Withdrawal,76.50,"2,868.15",0.00,0,234.21,"3,102.36",Whittier,90602,33.97,-118.02,Churned,Yes,1,Price,Extra data charges,26265,2021-08-01,3.08,60+


## Entregable — exportar `telco_cleaned.csv`

Exportamos el dataset integrado, limpio y con las variables derivadas a
`data/cleaned/telco_cleaned.csv`. Este será el **punto de partida único y
compartido** para el análisis multivariante (PASO 3), el modelado predictivo
(PASO 4) y el clustering (PASO 5).

In [18]:
import os
os.makedirs(CLEANED_PATH, exist_ok=True)

output_file = CLEANED_PATH + 'telco_cleaned.csv'
telco_df.to_csv(output_file, index=False)

print(f"Guardado: {output_file}")
print(f"Shape: {telco_df.shape[0]} filas x {telco_df.shape[1]} columnas")

# Verificacion de relectura
check = pd.read_csv(output_file)
print(f"\nVerificacion de relectura: {check.shape[0]} filas x {check.shape[1]} columnas")
print("Nulos tras relectura:", check.isnull().sum().sum())

Guardado: ../data/cleaned/telco_cleaned.csv
Shape: 7043 filas x 48 columnas

Verificacion de relectura: 7043 filas x 48 columnas
Nulos tras relectura: 0


## Preguntas de reflexión — ETL

### 1. ¿Cuántas filas tiene el dataset final integrado? ¿Coincide con el número de clientes únicos que encontraste en el EDA?

`telco_cleaned.csv` tiene **7.043 filas y 48 columnas**. Coincide exactamente con
los **7.043 clientes únicos** identificados en el EDA: ningún cliente se ha perdido
ni duplicado durante el `merge` (lo verificamos comprobando que `inner` y `left`
producen el mismo número de filas, y que el merge no introdujo ningún `NaN` nuevo).

### 2. ¿Por qué es importante eliminar columnas que contengan directamente información de churn (como "churn_reason") antes de hacer cualquier modelo predictivo?

Porque son un caso de **data leakage directo**: `churn_reason`,
`churn_category` y `churn_label` son **consecuencia** de que el cliente ya ha hecho
churn — son información que **solo existe después del hecho que queremos
predecir**. Si se incluyeran como variables predictoras (`X`), el modelo no
"aprendería a predecir el churn", sino que básicamente **leería la respuesta
directamente en una de las columnas** (p. ej., `churn_reason != 'No Churn'`
implica casi siempre `churn_value = 1`). El resultado serían métricas
artificialmente perfectas (AUC cercano a 1) en entrenamiento y test, pero el modelo
sería **inútil en producción**, porque para un cliente activo real `churn_reason`
siempre será `'No Churn'` (no se conoce el motivo de un churn que no ha ocurrido
todavía) — el modelo no tendría ninguna señal real que explotar.

**Importante:** en `telco_cleaned.csv` **mantenemos** estas columnas a propósito,
porque:
- El PASO 3 (análisis multivariante) necesita `churn_value` para calcular
  correlaciones y Cramér's V.
- El PASO 6 (dashboard) necesita `customer_status`, `churn_category` y
  `churn_reason` para la "Vista 2 — Análisis de churn".

La eliminación de columnas de leakage es una decisión que se toma **en el PASO 4**,
de forma explícita y documentada, justo antes de construir `X` (el conjunto de
variables predictoras) — no en la ETL, que sirve a múltiples consumidores.

### 3. ¿Qué tipo de join elegiste para integrar las tablas y por qué?

**`left join`** en los dos merges (4 tablas de cliente por `customer_id`, y
`population` por `zip_code`), tomando `demographics` como tabla base.

En este dataset concreto, **`inner` y `left` producen exactamente el mismo
resultado** (7.043 filas), porque las 4 tablas de cliente comparten el mismo
conjunto de `customer_id` y todos los `zip_code` de `location` existen en
`population`. Aun así, elegimos `left` porque:

- Es la opción **defensiva**: si en el futuro llegara una nueva carga de datos en la
  que, por ejemplo, un cliente nuevo aún no tuviera fila en `status` (porque el
  equipo de retención todavía no ha registrado su estado), un `inner join`
  **eliminaría ese cliente sin avisar**. Con `left`, el cliente se mantiene y el
  problema se hace visible como un `NaN` que hay que decidir cómo tratar.
- Responde directamente a la pregunta del propio documento ("¿qué pasaría si uso un
  inner join en lugar de left join? ¿Perderías clientes?"): en los datos actuales,
  **no perderíamos clientes con ninguno de los dos**, pero `left` es la opción que
  **garantiza** que no se pierdan de forma silenciosa si los datos cambian.

### 4. ¿Qué decisiones de imputación tomaste y cómo las justificarías ante el negocio?

Todas las imputaciones realizadas son de **categoría de negocio**, no estadísticas
(no se ha usado media, mediana ni moda en ningún caso):

| Columna | Decisión | Justificación para negocio |
|---|---|---|
| `offer` | `NaN → 'No Offer'` | "Vacío" en esta columna significa que el cliente no tiene ninguna promoción activa, no que falte el dato. Codificarlo como su propia categoría permite comparar la tasa de churn de "sin oferta" contra cada oferta concreta. |
| `internet_type` | `NaN → 'No Internet'` | Coincide al 100% con `internet_service = 'No'`. Un cliente sin internet no puede tener un "tipo de internet"; es una categoría válida, no un dato perdido. |
| `churn_category` | `NaN → 'No Churn'` | Solo los clientes que han hecho churn tienen una categoría de motivo. Para el resto, `'No Churn'` es la respuesta correcta, no una estimación. |
| `churn_reason` | `NaN → 'No Churn'` | Mismo razonamiento que `churn_category`. |

Además, aplicamos dos **correcciones de inconsistencia** que también podrían
plantearse a negocio, aunque no son "imputaciones" en sentido estricto:

- `gender`: unificamos `'M'/'F'` → `'Male'/'Female'` (mismo significado, dos
  formatos).
- `under_30` / `senior_citizen`: recalculados a partir de `age` para que sean
  siempre coherentes (afecta a 109 clientes con edades anómalas, ver pregunta 5).
- `monthly_charge`: corregido el signo en 120 clientes (`abs()`).

Si tuviera que defender estas decisiones ante Amalia, el mensaje sería: **"no hemos
inventado ningún dato — solo hemos hecho explícito lo que el propio dataset ya
decía implícitamente (sin oferta, sin internet, sin motivo de churn porque no hay
churn), y hemos corregido 3 inconsistencias de formato/signo que afectan a un
porcentaje pequeño de clientes (1,5%-1,7%) y que probablemente vienen de un error
de carga"**.

### 5. ¿Quedaron nulos en el dataset final? ¿En qué columnas? ¿Es un problema para los siguientes pasos?

**No.** El dataset final `telco_cleaned.csv` tiene **0 valores nulos** en sus 48
columnas (lo hemos verificado explícitamente tras el merge y tras la imputación, y
de nuevo al releer el CSV exportado).

Esto **simplifica mucho** los siguientes pasos:

- **PASO 3 (multivariante):** el cálculo de VIF y Cramér's V no requiere ningún
  filtrado adicional por nulos.
- **PASO 4 (predictivo) y PASO 5 (clustering):** los pipelines de `sklearn` incluyen
  un `SimpleImputer` "por si acaso" (buena práctica si en el futuro entran datos
  nuevos con huecos), pero **no tendrá nada que imputar** sobre
  `telco_cleaned.csv` tal y como está ahora.

La única salvedad **no es un nulo, sino una limitación de calidad documentada**:
los 109 clientes con `age` entre 101 y 119 (sección 5.2) siguen en el dataset con
ese valor de edad. No es un `NaN`, así que no bloquea ningún pipeline, pero
**vigilaremos su efecto** como posible outlier en la matriz de correlación (PASO 3)
y en la visualización PCA (PASO 5).